In [3]:
"""
UPV Formación Permanente Knowledge Base Extractor
=================================================
"""

# ─────────────────────────────────────────────
# CELDA 1
# ─────────────────────────────────────────────

!pip install -q beautifulsoup4 markdownify requests


# ─────────────────────────────────────────────
# CELDA 2
# ─────────────────────────────────────────────

import os
import re
import json
import time
import pickle
import requests
import markdownify

from bs4 import BeautifulSoup
from google.colab import drive


drive.mount('/content/drive')


JSON_URL = (
    "/content/drive/MyDrive/TFG Teleco/JSONs/"
    "formacion_permanente_upv.json"
)

PATH_KB = (
    "/content/drive/MyDrive/TFG Teleco/"
    "UPV_FormacionPermanente_KB"
)

ESTADO = os.path.join(PATH_KB, "_estado.pkl")

PAUSA = 0.4

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; UPV-KB-Bot/2.0)"
}

os.makedirs(PATH_KB, exist_ok=True)


# ─────────────────────────────────────────────
# CELDA 3
# ─────────────────────────────────────────────

def get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        time.sleep(PAUSA)
        return r.text
    except Exception as e:
        print("Error:", url, e)
        return None


def cargar_estado():
    if os.path.exists(ESTADO):
        with open(ESTADO, "rb") as f:
            return pickle.load(f)
    return set()


def guardar_estado(s):
    with open(ESTADO, "wb") as f:
        pickle.dump(s, f)


# ─────────────────────────────────────────────
# CELDA 4
# ─────────────────────────────────────────────

def buscar_patron(texto, patrones):
    for p in patrones:
        m = re.search(p, texto, re.I)
        if m:
            return m.group(1).strip()
    return None


def extraer_metadata_mejorada(texto):

    meta = {}

    meta["precio"] = buscar_patron(texto, [
        r"Precio\s+(\d+[.,]?\d*\s*€)",
        r"(\d+[.,]?\d*)\s*€"
    ])

    meta["horas"] = buscar_patron(texto, [
        r"(\d+)\s*h\b",
        r"(\d+)\s*horas"
    ])

    meta["ects"] = buscar_patron(texto, [
        r"(\d+)\s*ECTS"
    ])

    # modalidad (robusto)
    meta["modalidad"] = None
    for m in ["Online", "Presencial", "Semipresencial", "Emisión en directo"]:
        if re.search(r"\b" + re.escape(m) + r"\b", texto, re.I):
            meta["modalidad"] = m
            break

    meta["fechas"] = buscar_patron(texto, [
        r"Desde:\s*(.+)",
        r"Hasta:\s*(.+)"
    ])

    meta["campus"] = buscar_patron(texto, [
        r"Campus\s+de\s+([A-Za-zÁÉÍÓÚáéíóú]+)"
    ])

    # responsable mejorado
    meta["responsable"] = buscar_patron(texto, [
        r"Responsable de la actividad\s*:?\s*\n(.+)"
    ])

    meta["promotor"] = buscar_patron(texto, [
        r"Promovido por:\s*(.+)",
        r"Organiza:\s*(.+)"
    ])

    return meta


# ─────────────────────────────────────────────
# CELDA 5
# ─────────────────────────────────────────────

def html_a_markdown_limpio(html):

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "form"]):
        tag.decompose()

    basura_textos = [
        "Suscríbete", "Registrarse", "Iniciar sesión",
        "Toggle navigation", "Buscar formación",
        "Descarga en PDF", "Boletín"
    ]

    for tag in soup.find_all(["div", "section", "aside"]):
        txt = tag.get_text(" ", strip=True)
        if any(b.lower() in txt.lower() for b in basura_textos):
            tag.decompose()

    main = soup.find("main") or soup.body or soup

    for a in main.find_all("a"):
        a.replace_with(a.get_text(" ", strip=True))

    md = markdownify.markdownify(str(main), heading_style="ATX")

    md = re.sub(r"\n{3,}", "\n\n", md)

    return md.strip()


# ─────────────────────────────────────────────
# EXTRACCIÓN DE SECCIONES REALES (CLAVE)
# ─────────────────────────────────────────────

def extraer_secciones(html):

    soup = BeautifulSoup(html, "html.parser")

    secciones = {}

    headers = soup.find_all(["h2", "h3"])

    for h in headers:

        titulo = h.get_text(" ", strip=True).lower()

        if "dirigida a" in titulo:
            key = "Dirigido a"
        elif "objetivos" in titulo:
            key = "Objetivos"
        elif "temas" in titulo:
            key = "Contenidos"
        elif "evaluación" in titulo:
            key = "Evaluación"
        elif "metodología" in titulo:
            key = "Metodología"
        elif "requisitos" in titulo:
            key = "Requisitos"
        else:
            continue

        bloque = []

        for sib in h.find_next_siblings():

            if sib.name in ["h2", "h3"]:
                break

            txt = sib.get_text(" ", strip=True)

            if txt:
                bloque.append(txt)

        if bloque:
            secciones[key] = " ".join(bloque)

    return secciones


# ─────────────────────────────────────────────
# CELDA 6
# ─────────────────────────────────────────────

def crear_markdown(item, html):

    soup = BeautifulSoup(html, "html.parser")
    texto = soup.get_text("\n", strip=True)

    meta = extraer_metadata_mejorada(texto)
    secciones = extraer_secciones(html)

    md = []

    md.append(f"# {item['nombre']}\n")

    md.append("## Información principal\n")

    campos = [
        ("Precio", "precio"),
        ("Horas", "horas"),
        ("ECTS", "ects"),
        ("Modalidad", "modalidad"),
        ("Fechas", "fechas"),
        ("Campus", "campus"),
        ("Responsable", "responsable"),
        ("Promueve", "promotor")
    ]

    for k, v in campos:
        if meta.get(v):
            md.append(f"- **{k}:** {meta[v]}")

    md.append(f"\nURL: {item['url']}\n")
    md.append("\n---\n")

    # SECCIONES ESTRUCTURADAS (LO IMPORTANTE)
    orden = [
        "Dirigido a",
        "Objetivos",
        "Contenidos",
        "Evaluación",
        "Metodología",
        "Requisitos"
    ]

    for o in orden:
        if o in secciones:
            md.append(f"\n## {o}\n")
            md.append(secciones[o])

    # fallback: contenido limpio general
    md.append("\n## Información adicional\n")
    md.append(html_a_markdown_limpio(html))

    return "\n".join(md)


# ─────────────────────────────────────────────
# CELDA 7
# ─────────────────────────────────────────────

with open(JSON_URL, encoding="utf-8") as f:
    datos = json.load(f)

formaciones = datos["formaciones"]

procesados = cargar_estado()

pendientes = [
    f for f in formaciones
    if f["id"] not in procesados
]

print("Pendientes:", len(pendientes))


for i, item in enumerate(pendientes, 1):

    print(f"[{i}/{len(pendientes)}] {item['nombre']}")

    html = get(item["url"])
    if not html:
        continue

    md = crear_markdown(item, html)

    nombre = re.sub(r"[^a-z0-9]+", "_", item["id"]) + ".md"
    ruta = os.path.join(PATH_KB, nombre)

    with open(ruta, "w", encoding="utf-8") as f:
        f.write(md)

    procesados.add(item["id"])

    if i % 10 == 0:
        guardar_estado(procesados)

guardar_estado(procesados)

print("✅ KB terminada:", PATH_KB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pendientes: 288
[1/288] ANÁLISIS DE LA COYUNTURA ECONÓMICA
[2/288] MODELOS MULTICRITERIO APLICADOS A LA GESTIÓN DE CARTERAS
[3/288] GESTIÓN DE CARTERAS II
[4/288] INCENDIOS DE ORIGEN ELÉCTRICO EN EL ÁMBITO DOMÉSTICO. CAUSAS, RIESGOS Y ACTUACIÓN
[5/288] CAMPOS MAGNÉTICOS EN INSTALACIONES ELÉCTRICAS Y ALREDEDORES, Y SU CÁLCULO Y REPRESENTACIÓN CON CRMAG PLUS
[6/288] ANÁLISIS Y DISEÑO DE PUESTAS A TIERRA EN INSTALACIONES ELÉCTRICAS CON CRGROUND®
[7/288] ASESOR FINANCIERO
[8/288] AGENTE FINANCIERO EUROPEO
[9/288] ASISTENTE FINANCIERO EUROPEO
[10/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN ASESORÍA FINANCIERA 2025
[11/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN CRÉDITO INMOBILIARIO 2025
[12/288] ASESOR FINANCIERO EN CRÉDITO HIPOTECARIO
[13/288] INFORMADOR FINANCIERO EN CRÉDITO HIPOTECARIO
[14/288] CLOUD COMPUTING CON AMAZON WEB SERVICES (AWS)
[15/288] BASES DE DATOS ESPACIA